In [0]:
import requests
import json
import logging
from datetime import datetime
from pyspark.sql.functions import lit, current_timestamp
from dataclasses import dataclass

# Setup basic logging instead of prints
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

@dataclass
class IngestionConfig:
    entity: str  
    catalog: str = "workspace"
    schema: str = "br_legislative"  
    volume: str = "landing_zone"
    
    @property
    def landing_path(self):
        return f"/Volumes/{self.catalog}/{self.schema}/{self.volume}/{self.entity}"
    
    @property
    def table_name(self):
        return f"{self.catalog}.{self.schema}.bronze_{self.entity}"

def ingest_to_bronze(config: IngestionConfig) -> None:
    """Fetches data from Camara API and lands it into Bronze Delta Table."""
    
    logger.info(f"Starting ingestion job for: {config.entity}")
    
    # 1. Ensure volume exists via Databricks SQL (Fixes Errno 95)
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {config.catalog}.{config.schema}.{config.volume}")
    
    # 2. API Request
    url = f"https://dadosabertos.camara.leg.br/api/v2/{config.entity}"
    response = requests.get(url, params={"formato": "json"})
    
    if response.status_code != 200:
        logger.error(f"Failed to fetch data from {url}. Status: {response.status_code}")
        raise Exception(f"API Error: {response.status_code}")
        
    data = response.json().get('dados', [])
    
    # 3. Create subfolder for the entity using dbutils (native Databricks way)
    dbutils.fs.mkdirs(config.landing_path)
    
    # 4. Save Raw JSON
    file_name = f"{config.entity}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    file_path = f"{config.landing_path}/{file_name}"
    
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)
        
    logger.info(f"Raw file landed successfully at {file_path}")
        
    # 5. Process to Bronze Delta
    df = spark.createDataFrame(data)
    df = df.withColumn("source_file", lit(file_path)) \
           .withColumn("ingestion_timestamp", current_timestamp())
    
    df.write.format("delta").mode("append").saveAsTable(config.table_name)
    logger.info(f"Bronze table {config.table_name} updated successfully.")

In [0]:
# Execute ingestion for deputies
try:
    config_dep = IngestionConfig(entity="deputados")
    ingest_to_bronze(config_dep)
except Exception as e:
    logger.error(f"Pipeline failed: {str(e)}")